In [7]:
import ollama

def basic_chat(prompt: str, model: str = "qwen3:8b") -> str:
    """
    最基本的调用方式：发送一条消息，等待完整回复。

    核心概念：
    - model: 使用哪个模型（ollama list 查看已安装的）
    - messages: 对话历史，格式为 [{"role": "user/assistant/system", "content": "..."}]
    - response["message"]["content"]: 模型的回复文本
    """
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]

In [8]:
basic_chat("你好，简单介绍一下自己")


'你好！我是通义千问，由通义实验室研发的超大规模语言模型。我能够理解并生成多种语言，擅长处理各种文本任务，比如回答问题、创作文字、逻辑推理、编程、数据分析等。我的训练数据截止到2024年4月，这意味着我可以提供最新的信息和知识。如果你有任何问题或需要帮助，随时告诉我！😊'

In [12]:
MODEL = "qwen3:8b"


In [10]:
def stream_chat(prompt: str, model: str = "qwen3:8b"):
    """
    流式调用：模型边生成边输出，不用等全部生成完。

    核心概念：
    - stream=True: 启用流式模式
    - 返回的是一个迭代器，每次 yield 一小段文本
    - 适合长回答或需要实时显示的场景
    """
    print(f"\n{'='*50}")
    print(f"[流式输出] 问题: {prompt}")
    print(f"{'='*50}")

    stream = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    full_response = ""
    for chunk in stream:
        text = chunk["message"]["content"]
        print(text, end="", flush=True)
        full_response += text

    print()  # 换行
    return full_response


# ============================================================
# 3. 多轮对话 - 维护对话历史
# ============================================================
def multi_turn_demo(model: str = "qwen3:8b"):
    """
    多轮对话的关键：把之前的 messages 都传给模型。

    核心概念：
    - 模型本身是无状态的，每次调用都是独立的
    - 要实现"记忆"，必须手动维护 messages 列表
    - 每次把完整的对话历史发送给模型
    - messages 越长，消耗的 tokens 越多（注意 context window 限制）
    """
    print(f"\n{'='*50}")
    print("[多轮对话演示]")
    print(f"{'='*50}")

    messages = []

    # 可选：设置 system prompt 定义助手角色
    messages.append(
        {
            "role": "system",
            "content": "你是一个友好的AI助手，回答简洁明了，用中文回答。",
        }
    )

    # 模拟3轮对话
    questions = [
        "Python 的 GIL 是什么？一句话解释",
        "那它对多线程有什么影响？",  # 这个问题依赖上文的"它"
        "有什么替代方案？",  # 继续追问
    ]

    for q in questions:
        print(f"\n用户: {q}")
        messages.append({"role": "user", "content": q})

        response = ollama.chat(model=model, messages=messages)
        answer = response["message"]["content"]

        # 把助手回复也加入历史，这样下一轮模型能看到
        messages.append({"role": "assistant", "content": answer})
        print(f"助手: {answer}")

    print(f"\n[对话历史共 {len(messages)} 条消息]")

In [13]:
stream_chat("介绍一下 RAG（检索增强生成）的基本原理", MODEL)



[流式输出] 问题: 介绍一下 RAG（检索增强生成）的基本原理
RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合**检索系统**和**生成模型**的混合方法，旨在通过引入外部知识库来提升生成模型的准确性和可靠性。其核心思想是：**在生成回答时，先从外部知识库中检索相关信息，再基于这些信息生成更准确、更丰富的回答**。以下是其基本原理和关键组成部分：

---

### **1. 核心思想**
RAG的核心是**将检索和生成两个模块结合**，通过以下方式增强生成模型的能力：
- **检索模块**：从外部知识库中找到与用户问题相关的信息。
- **生成模块**：利用检索到的信息，结合生成模型（如Transformer）生成最终答案。

---

### **2. 关键组成部分**
#### **(1) 检索器（Retrieval System）**
- **功能**：从大规模文档集合（如维基百科、论文、数据库等）中检索与用户问题相关的**相关文档**。
- **实现方式**：
  - 使用**向量检索**（如FAISS、HNSW）或**基于关键词的检索**（如BM25）。
  - 通过**语义相似性**（如余弦相似度）匹配用户查询和文档内容。
- **输出**：一组与用户问题相关的**候选文档**（或文档片段）。

#### **(2) 生成器（Generation Model）**
- **功能**：基于检索到的信息，生成自然流畅的回答。
- **实现方式**：
  - 使用**预训练的语言模型**（如GPT、BERT、T5等）。
  - 将检索到的文档内容与用户问题作为**上下文输入**，生成最终答案。
- **输出**：结合检索信息的**生成式回答**。

---

### **3. 工作流程**
RAG的典型流程分为以下步骤：
1. **用户输入**：用户提出一个需要回答的问题。
2. **检索阶段**：
   - 将用户问题转化为查询向量或关键词。
   - 在知识库中检索最相关的文档或段落。
3. **生成阶段**：
   - 将检索到的文档内容与用户问题拼接为上下文。
   - 输入到生成模型中，生成最终答案。
4. **输出答案**：将生成的答案返回给用户。

---

### **4. 优势**
- *

'RAG（Retrieval-Augmented Generation，检索增强生成）是一种结合**检索系统**和**生成模型**的混合方法，旨在通过引入外部知识库来提升生成模型的准确性和可靠性。其核心思想是：**在生成回答时，先从外部知识库中检索相关信息，再基于这些信息生成更准确、更丰富的回答**。以下是其基本原理和关键组成部分：\n\n---\n\n### **1. 核心思想**\nRAG的核心是**将检索和生成两个模块结合**，通过以下方式增强生成模型的能力：\n- **检索模块**：从外部知识库中找到与用户问题相关的信息。\n- **生成模块**：利用检索到的信息，结合生成模型（如Transformer）生成最终答案。\n\n---\n\n### **2. 关键组成部分**\n#### **(1) 检索器（Retrieval System）**\n- **功能**：从大规模文档集合（如维基百科、论文、数据库等）中检索与用户问题相关的**相关文档**。\n- **实现方式**：\n  - 使用**向量检索**（如FAISS、HNSW）或**基于关键词的检索**（如BM25）。\n  - 通过**语义相似性**（如余弦相似度）匹配用户查询和文档内容。\n- **输出**：一组与用户问题相关的**候选文档**（或文档片段）。\n\n#### **(2) 生成器（Generation Model）**\n- **功能**：基于检索到的信息，生成自然流畅的回答。\n- **实现方式**：\n  - 使用**预训练的语言模型**（如GPT、BERT、T5等）。\n  - 将检索到的文档内容与用户问题作为**上下文输入**，生成最终答案。\n- **输出**：结合检索信息的**生成式回答**。\n\n---\n\n### **3. 工作流程**\nRAG的典型流程分为以下步骤：\n1. **用户输入**：用户提出一个需要回答的问题。\n2. **检索阶段**：\n   - 将用户问题转化为查询向量或关键词。\n   - 在知识库中检索最相关的文档或段落。\n3. **生成阶段**：\n   - 将检索到的文档内容与用户问题拼接为上下文。\n   - 输入到生成模型中，生成最终答案。\n4. **输出答案**：将生成的答案返回给用户。\n\n---\n\n### **4. 

In [14]:
multi_turn_demo(MODEL)



[多轮对话演示]

用户: Python 的 GIL 是什么？一句话解释
助手: Python的GIL（全局解释器锁）是CPython解释器中用于同步线程的机制，确保同一时间只有一个线程执行Python字节码，从而避免多线程并发操作导致的数据不一致问题。

用户: 那它对多线程有什么影响？
助手: Python的GIL会限制多线程的并行执行，导致**CPU密集型任务无法充分利用多核CPU**，但**I/O密集型任务仍能通过线程切换提高效率**。

用户: 有什么替代方案？
助手: Python的GIL限制了多线程的并行执行，但可通过以下方式绕过或优化：  
1. **多进程**（`multiprocessing`）：用多个进程替代线程，每个进程拥有独立的Python解释器和内存空间，可充分利用多核CPU。  
2. **异步IO（async/await）**：通过事件循环实现非阻塞I/O操作，适合I/O密集型任务（如网络请求、文件读写）。  
3. **C扩展/第三方库**：用C/C++/Rust等语言编写核心逻辑（如NumPy、Pandas），绕过GIL限制。  
4. **多线程+队列**：用线程处理I/O任务，通过队列传递数据，避免直接操作共享资源。  
5. **使用无GIL的Python解释器**（如PyPy）：部分解释器对GIL的处理更灵活，但需注意兼容性。

[对话历史共 7 条消息]


In [ ]:
# ============================================================
# 5. 调节参数 - Temperature, num_ctx 等
# ============================================================
def parameter_demo(model: str = "qwen3:8b"):
    """
    常用参数说明：

    - temperature: 控制随机性 (0=确定性, 1=较随机, 2=很随机)
      RAG 任务建议 0.1-0.3，创意任务 0.7+
    - num_ctx: 上下文窗口大小（token数）
      Qwen3:8B 最大支持 128k，但本地建议 4096-8192
    - top_p: nucleus sampling，和 temperature 配合使用
    - seed: 设置随机种子，相同 seed 输出一致（可复现）
    """
    print(f"\n{'='*50}")
    print("[Temperature 参数对比]")
    print(f"{'='*50}")

    prompt = "/no_think 用一句话描述春天"

    for temp in [0.1, 0.7, 1.5]:
        print(f"\ntemperature={temp}:")
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": temp,
                "num_ctx": 4096,  # 设置上下文窗口
            },
        )
        print(f"  {response['message']['content']}")